# 07 — Pharmacogenomic candidate evidence foundations

Research-only; not a clinical call, validated assay, prescribing engine, or recommendation. Personal mode requires a completed private M2 run, checked local bundle, and explicit genes. Colab uses a Google-managed VM; use the offline local workflow when cloud processing is unacceptable.


In [ ]:
import os
import sys

PROFILE = "synthetic_ci"
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = "main"
WORKSPACE_ROOT = "/content/drive/MyDrive/genome-evidence-private"
SUBJECT_ID = "subject-0001"
SELECTED_GENES = []
SHOW_PRIVATE_RESULTS = False
PROFILE = os.environ.get("GENOME_EVIDENCE_PROFILE", PROFILE)

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    loaded_package_modules = [
        module
        for name, module in sys.modules.items()
        if name == "genome_evidence" or name.startswith("genome_evidence.")
    ]
    if loaded_package_modules:
        current_commit = subprocess.run(
            ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD^{commit}"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        loaded_paths = [
            Path(str(module_file)).resolve()
            for module in loaded_package_modules
            if (module_file := getattr(module, "__file__", None)) is not None
        ]
        if current_commit != RESOLVED_COMMIT or any(
            not path.is_relative_to(CHECKOUT.resolve()) for path in loaded_paths
        ):
            raise RuntimeError(
                "genome_evidence modules from another revision are already loaded; "
                "restart the runtime and rerun from the first cell"
            )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from pathlib import Path

from genome_evidence.pharmacogenomics import GeneOutcome

if PROFILE == "personal_drive":
    from genome_evidence.workspace import validate_workspace

    workspace = validate_workspace(Path(WORKSPACE_ROOT))
    if not SELECTED_GENES:
        raise RuntimeError("Select one or more genes explicitly; no synthetic fallback is allowed")
    bundles = sorted((workspace / "references/pharmacogenomics").glob("*/bundle_manifest.json"))
    if len(bundles) != 1:
        raise RuntimeError("Exactly one checked local M8 bundle is required")
else:
    # Fabricated status contract demonstration; package unit/integration tests execute matching.
    synthetic_statuses = {
        "fabricated-resolved": GeneOutcome.RESOLVED_CANDIDATE,
        "fabricated-phase-ambiguous": GeneOutcome.AMBIGUOUS_CANDIDATES,
        "fabricated-missing": GeneOutcome.INSUFFICIENT_COVERAGE,
        "fabricated-unmodeled": GeneOutcome.UNMODELED_OBSERVED_VARIATION,
        "fabricated-structural": GeneOutcome.UNSUPPORTED_GENE_OR_METHOD,
    }
    assert set(synthetic_statuses.values()) == {
        GeneOutcome.RESOLVED_CANDIDATE,
        GeneOutcome.AMBIGUOUS_CANDIDATES,
        GeneOutcome.INSUFFICIENT_COVERAGE,
        GeneOutcome.UNMODELED_OBSERVED_VARIATION,
        GeneOutcome.UNSUPPORTED_GENE_OR_METHOD,
    }

## Aggregate output and next action

Validate coverage, evaluability, ambiguity, unsupported methods, artifact schemas, hashes, and completion before inspecting private results. Candidate diplotypes and source-attributed phenotype evidence remain hidden unless `SHOW_PRIVATE_RESULTS = True`; enabling it may persist/share notebook output and does not create a clinical result. Missing loci never become reference or `*1`. Obtain an appropriate validated laboratory assay and qualified clinician/pharmacist review before any medication decision.
